> **INSTRUCTOR SOLUTIONS** — do not share with learners before the session.

# Part 12 · Notebook 03 — Spread-aware orders, execution algorithms and TCA

**Sessions:** S5–S6 (Spread-aware order management · Execution algorithms & TCA) · [Lesson plan](../../docs/lessons/PART_12_TRADING_PLATFORM_CAPSTONE.md) · graded labs in [`labs/part12/`](../../labs/part12/)

**You will:**
1. Price a limit order inside the spread and round it to the tick in the safe direction.
2. Simulate passive, mid and crossing orders on a quote path with informed flow, counting non-fills.
3. Measure implementation shortfall against the decision price.
4. Compare a single order with TWAP, VWAP and POV on a 60,000-share parent order.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
Everything runs offline on synthetic data with known answers (planted violations, injected faults, a strategy that stops working): no broker, Docker or network is needed.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p12lib.py is in notebooks/part12/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p12lib as p

p.use_course_style()

## 1. Limit prices on the tick

BUY at `bid + a·(ask − bid)`, SELL at `ask − a·(ask − bid)`, where `a` is the aggressiveness (0 passive, 1 cross). Round to the tick with `Decimal`, in the direction that never overpays: a BUY rounds **down** (`ROUND_DOWN`), a SELL rounds **up** (`ROUND_UP`). `(Decimal(str(px)) / q).to_integral_value(rounding=...)` gives the number of ticks.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
from decimal import ROUND_DOWN, ROUND_UP, Decimal

def limit_price(side, bid, ask, tick, aggressiveness=0.0):
    px = bid + aggressiveness * (ask - bid) if side == "BUY" else ask - aggressiveness * (ask - bid)
    q = Decimal(str(tick))
    steps = (Decimal(str(px)) / q).to_integral_value(rounding=ROUND_DOWN if side == "BUY" else ROUND_UP)
    return float(steps * q)

cases = [("BUY", 100.00, 100.05, 0.01, 0.0), ("BUY", 100.00, 100.05, 0.01, 0.33), ("SELL", 100.00, 100.05, 0.01, 0.33),
         ("BUY", 100.00, 100.05, 0.01, 1.0), ("SELL", 4.10, 4.15, 0.05, 0.5), ("BUY", 4.10, 4.15, 0.05, 0.5)]
mine = [p.attempt(limit_price, *c) for c in cases]
mine = p.check("limit_price", mine, [p.limit_price(*c) for c in cases])
pd.DataFrame(cases, columns=["side", "bid", "ask", "tick", "aggressiveness"]).assign(limit=mine)

## 2. Passive orders look cheap until you count non-fills

`p.quote_path` has **informed** order flow: a passive buy tends to fill when sellers are pushing the price down, and goes unfilled when the price runs away. `p.chase_report` sends 150 random orders per path under each schedule of aggressiveness levels (5 seconds each) and measures the fill rate, the cost of filled orders, the **markout** (the mid 30 s after the fill vs the fill price: negative = adverse selection), and the **all-in** cost, which charges an unfilled order the cost of crossing at the end. Averaged over three quote paths:

In [ ]:
schedules = {"passive": (0.0,), "mid": (0.5,), "cross": (1.0,), "chase passive → cross": (0.0, 0.33, 0.67, 1.0)}
sweep = sum(p.chase_report(p.quote_path(seed=s), schedules, 150, seed=s) for s in range(3)) / 3
sweep.round(2)

Passive orders look cheap when they fill (about −1.7 bps: they earn part of the spread), but the markout shows about half of that saving gone 30 seconds later: they fill when the price is moving against them (it keeps falling after a passive buy fills). They fill only about two times in three, and completing the rest costs the spread and then some. Measured all-in, the chase schedule is the cheapest and crossing at once the most expensive. Your backtest's cost model should use these measured numbers.

## 3. Implementation shortfall

For a parent order, TCA compares the average fill price with the **decision** price: `IS (bps) = 1e4 · (sign·(avg − decision)·qty + fees) / (decision·qty)`, with `sign = +1` for BUY, −1 for SELL, `qty` the total filled and `avg` the quantity-weighted average price. Positive = cost.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def implementation_shortfall_bps(side, decision_px, fills, fees=0.0):
    qty = sum(q for _, q in fills)
    avg = sum(px * q for px, q in fills) / qty
    sign = 1 if side == "BUY" else -1
    return 1e4 * (sign * (avg - decision_px) * qty + fees) / (decision_px * qty)

cases = [("BUY", 100.0, [(100.10, 300), (100.20, 700)]), ("SELL", 100.0, [(99.90, 1000)], 5.0), ("BUY", 50.0, [(49.95, 100)])]
mine = [p.attempt(implementation_shortfall_bps, *c) for c in cases]
mine = p.check("implementation_shortfall_bps", mine, [p.implementation_shortfall_bps(*c) for c in cases])
print(np.round(mine, 2))

## 4. Slice big orders

Buy 60,000 shares in a session whose volume is heavy at the open and close. Each child fill pays a temporary impact proportional to its share of that minute's volume (`p.execute_schedule`). Compare: everything in the first minute; TWAP (13 equal slices); VWAP (sized by the **previous** session's 30-minute volume profile, known before the open); POV (10% of the previous minute's volume until done). Five sessions:

In [ ]:
def pov_schedule(qty, session, participation=0.1):
    remaining, out, vols = qty, [], session["volume"].to_numpy()
    for i in range(len(session)):
        child = 0 if i == 0 else p.pov_child_qty(vols[i - 1], participation, remaining)
        remaining -= child
        out.append(child)
    return pd.Series(out, index=session.index)

qty, results = 60_000, {k: [] for k in ("single", "TWAP", "VWAP", "POV 10%")}
for s in range(1, 6):
    sess, prev = p.intraday_session(s), p.intraday_session(s - 1)
    start, end = sess.index[0], sess.index[-1] + pd.Timedelta(minutes=1)
    profile = prev["volume"].resample("30min").sum()
    profile.index = pd.date_range(start, periods=len(profile), freq="30min")
    schedules = {"single": pd.Series([qty], index=[start]), "TWAP": p.twap_schedule(qty, start, end, 13),
                 "VWAP": p.vwap_schedule(qty, profile), "POV 10%": pov_schedule(qty, sess)}
    for name, sched in schedules.items():
        results[name].append(p.implementation_shortfall_bps("BUY", sess["price"].iloc[0], p.execute_schedule("BUY", sched, sess)))
pd.DataFrame({"mean IS (bps)": {k: np.mean(v) for k, v in results.items()}, "std (bps)": {k: np.std(v, ddof=1) for k, v in results.items()}}).round(1)

One big order pays enormous impact. TWAP and VWAP cut it by more than two thirds, but they stay exposed to the price drifting all day, so their results vary a lot. POV finishes early in the heavy morning volume: less drift exposure, lower and steadier cost here. The right algo depends on urgency and on how much the price can move while you wait.

## Wrap-up

* Price inside the spread, round the safe way, and measure all-in cost including non-fills.
* Informed flow makes passive fills adverse; markouts reveal it.
* TCA against the decision price; feed the measured costs back into the backtest.
* Graded version: `labs/part12/week42_execution` (`limit_price`, `simulate_chase`, `chase_report`, the schedules, `implementation_shortfall_bps`) and Clinic W2 (the execution-quality report).